In [0]:
# ==============================================================================
# UNIVERSIDAD LATINOAMERICANA DE CIENCIA Y TECNOLOGÍA (ULACIT)
# Curso: Big Data y Tecnologías de Procesamiento
# Scripts de Ingestión y Almacenamiento (Capa Bronce y Plata - Medallion)
# ==============================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, regexp_replace
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col, to_date, regexp_replace


In [0]:
# 0. Inicialización 
PATH_ORIGEN_CSV = "/Volumes/workspace/bigdata/test/store_sales.csv"

print("=== CSV cargado correctamente ===")


=== CSV cargado correctamente ===


In [0]:
# ==============================================================================
# 1. CAPA DE INGESTA Y ALMACENAMIENTO BRONCE (Raw Data)
# Objetivo: Cargar los datos crudos
# ==============================================================================

# Definición del esquema original para asegurar una correcta lectura inicial en modo Batch
esquema_bronce = StructType([
    StructField("Transaction ID", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Item", StringType(), True),
    StructField("Price Per Unit", DoubleType(), True),
    StructField("Quantity", DoubleType(), True),
    StructField("Total Spent", DoubleType(), True),
    StructField("Payment Method", StringType(), True),
    StructField("Location", StringType(), True),
    StructField("Transaction Date", StringType(), True),
    StructField("Discount Applied", StringType(), True)
])

print("\n[INFO] Leyendo archivo CSV estático en modo Batch...")
df_raw = spark.read.format("csv").option("header", "true").schema(esquema_bronce).load(PATH_ORIGEN_CSV)


[INFO] Leyendo archivo CSV estático en modo Batch...


In [0]:
df_storesales = spark.read.csv(
    PATH_ORIGEN_CSV,
    header=True,
    schema=esquema_bronce,   # <-- en vez de inferSchema=True
    encoding="UTF-8"         # <-- explicito: el dataset tiene tildes (San Jose, Limon)
)

print(f"BigMart CR Sales cargado: {df_storesales.count():,} filas, {len(df_storesales.columns)} columnas")
df_storesales.printSchema()
display(df_storesales.limit(10))

BigMart CR Sales cargado: 5,000 filas, 11 columnas
root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Total Spent: double (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: string (nullable = true)
 |-- Discount Applied: string (nullable = true)



Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
1,58,Female,Accessories,null,115.5,null,Card,3.5,18,4
2,40,Male,Mens Clothing,null,103.43,null,Card,4.1,13,4
3,66,Female,Sports,null,35.45,null,Card,3.3,11,3
4,39,Female,Accessories,null,153.31,null,Card,4.4,13,4
5,23,Female,Home,null,151.43,null,Card,4.1,20,10
6,48,Male,Groceries,null,36.76,null,Card,2.5,14,2
7,54,Male,Electronics,null,1396.16,null,Card,4.7,18,4
8,59,Female,Beauty,null,35.55,null,Card,4.6,14,7
9,51,Male,Electronics,null,2238.79,null,Card,4.3,20,5
10,47,Female,Footwear,null,225.49,null,Card,4.7,18,4


In [0]:
#Mostrar el tipo de datos por columna
print(df_raw.dtypes)

[('Transaction ID', 'string'), ('Customer ID', 'string'), ('Category', 'string'), ('Item', 'string'), ('Price Per Unit', 'double'), ('Quantity', 'double'), ('Total Spent', 'double'), ('Payment Method', 'string'), ('Location', 'string'), ('Transaction Date', 'string'), ('Discount Applied', 'string')]


In [0]:
# Renombrar columnas para eliminar espacios (requerido por Delta Lake)
df_raw = df_raw.toDF(*[col_name.replace(" ", "") for col_name in df_raw.columns])

In [0]:
# Guardar en Capa Bronce usando formato Delta (estándar del enfoque Lakehouse)
print("[BRONCE] Guardando datos crudos en la Capa Bronce (Delta Lake)...")
df_raw.write.format("delta").mode("overwrite").saveAsTable("default.retail_sales_bronce")

print("[BRONCE] Tabla 'default.retail_sales_bronce' creada exitosamente.")

[BRONCE] Guardando datos crudos en la Capa Bronce (Delta Lake)...
[BRONCE] Tabla 'default.retail_sales_bronce' creada exitosamente.


In [0]:
from pyspark.sql.functions import to_date

# Convertir la fecha con la fecha de transacción a formato fecha y el porcentaje de descuento a porcentaje
df_datos_limpios = df_raw.withColumn(
    "TransactionDate",
    to_date(col("TransactionDate"), "yyyy-MM-dd")  # Cambia el formato de ambas columnas
).withColumn(
    "DiscountApplied",
    (
        regexp_replace(col("DiscountApplied"), "%", "").cast(DoubleType()) / 100
    )
)

# Verificar el esquema
df_datos_limpios.printSchema()

root
 |-- TransactionID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- PricePerUnit: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- TotalSpent: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- DiscountApplied: double (nullable = true)



In [0]:
# DEMO 1.1: Estadisticas numericas con describe()
print("BIGMART CR SALES — ESTADISTICAS NUMERICAS")
df_raw.select('Item', 'Location', 'Quantity', 'Category', 'CustomerID').describe().show()

BIGMART CR SALES — ESTADISTICAS NUMERICAS
+-------+---------------+-----------------+-----------------+--------+------------------+
|summary|           Item|         Location|         Quantity|Category|        CustomerID|
+-------+---------------+-----------------+-----------------+--------+------------------+
|  count|           5000|             5000|             5000|    5000|              5000|
|   mean|           NULL| 3.78415999999999|285.0905219999998|    NULL|           45.2248|
| stddev|           NULL|0.681796203870864| 551.454382174256|    NULL|14.564995460986442|
|    min|    Accessories|              1.1|             5.08|  Female|                20|
|    max|Womens Clothing|                5|          2997.94|    Male|                70|
+-------+---------------+-----------------+-----------------+--------+------------------+



In [0]:
from pyspark.sql.functions import count

total_rows = df_raw.count()

null_counts = df_raw.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_raw.columns
]).collect()[0].asDict()

print(f"\n{'Columna':<25}{'Nulos':<10}{'%':<8}")
for col_name, null_count in null_counts.items():
    pct = 100 * null_count / total_rows
    flag = "  <-- atencion" if pct > 0 else ""
    print(f"{col_name:<25}{null_count:<10}{pct:>5.2f}%{flag}")


Columna                  Nulos     %       
TransactionID            0          0.00%
CustomerID               0          0.00%
Category                 0          0.00%
Item                     0          0.00%
PricePerUnit             5000      100.00%  <-- atencion
Quantity                 0          0.00%
TotalSpent               5000      100.00%  <-- atencion
PaymentMethod            0          0.00%
Location                 0          0.00%
TransactionDate          0          0.00%
DiscountApplied          0          0.00%


In [0]:
cantidad_de_duplicados = df_raw.count() - df_raw.dropDuplicates().count()
print(f"\nDuplicados exactos (todas las columnas iguales): {cantidad_de_duplicados}")
print("Estrategia: DROP directo con dropDuplicates() -- son filas 100% redundantes.")


Duplicados exactos (todas las columnas iguales): 0
Estrategia: DROP directo con dropDuplicates() -- son filas 100% redundantes.


In [0]:
 # Distribucion y outliers en la columna Quanttity

percentiles = df_raw.approxQuantile('Quantity', [0.01, 0.25, 0.5, 0.75, 0.95], 0.01)

print(f"\nPercentil 1:   {percentiles[0]:>10.2f}")
print(f"Percentil 25:  {percentiles[1]:>10.2f}")
print(f"Mediana (50):  {percentiles[2]:>10.2f}")
print(f"Percentil 75:  {percentiles[3]:>10.2f}")
print(f"Percentil 95:  {percentiles[4]:>10.2f}")

outliers = df_raw.filter(col('Quantity') > percentiles[4]).count()
print(f"\nPosibles outliers (> percentil 95): {outliers} ({100*outliers/df_raw.count():.2f}%)")


Percentil 1:         5.08
Percentil 25:       70.08
Mediana (50):      120.93
Percentil 75:      182.00
Percentil 95:     1512.50

Posibles outliers (> percentil 95): 299 (5.98%)


In [0]:
from pyspark.sql.functions import (
    col, count, when, trim, isnan,
    min, max, avg, stddev, countDistinct
)
from pyspark.sql.types import StringType, NumericType

print("="*80)
print("PRUEBAS PRELIMINARES Y DE CALIDAD DE LOS DATOS")
print("="*80)

# 1. Dimensiones del DataFrame
print("\n1. DIMENSIONES DEL DATAFRAME")
print(f"Número de filas: {df_raw.count()}")
print(f"Número de columnas: {len(df_raw.columns)}")

PRUEBAS PRELIMINARES Y DE CALIDAD DE LOS DATOS

1. DIMENSIONES DEL DATAFRAME
Número de filas: 5000
Número de columnas: 11


In [0]:
# Verificar la cantidad de valores únicos
print("\n8. VALORES ÚNICOS POR COLUMNA")

for c in df_raw.columns:
    print(f"{c}: {df_raw.select(countDistinct(c)).first()[0]}")


8. VALORES ÚNICOS POR COLUMNA
TransactionID: 5000
CustomerID: 51
Category: 2
Item: 9
PricePerUnit: 0
Quantity: 4565
TotalSpent: 0
PaymentMethod: 2
Location: 39
TransactionDate: 32
DiscountApplied: 14
